In [ ]:
#@title 1. Fill in keys and options
OPENAI_API_KEY = ""  #@param {type:"string"}

SEARCH_ENGINE = "tavily"  #@param ["tavily", "perplexity", "perplexity-long", "brave", "exa"]
TAVILY_API_KEY = ""  #@param {type:"string"}
PERPLEXITY_API_KEY = ""  #@param {type:"string"}
BRAVE_API_KEY = ""  #@param {type:"string"}
EXA_API_KEY = ""  #@param {type:"string"}

MODEL = "gpt-5-nano"  #@param {type:"string"}
GRADER_MODEL = "gpt-5-nano"  #@param {type:"string"}
HF_DATASET_CONFIG = "verified"  #@param ["verified", "synthetic", "verified_with_metadata"]
DRY_RUN = True  #@param {type:"boolean"}
MAX_WORKERS = 1  #@param {type:"integer"}
RUN_SUFFIX = "colab-gpt-5-nano"  #@param {type:"string"}

REPO_URL = "https://github.com/prometheus-eval/K-BrowseComp.git"


In [ ]:
#@title 2. Clone the repo and install dependencies
import subprocess
import sys
from pathlib import Path

repo_dir = Path("/content/K-BrowseComp")
if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.12"], cwd=repo_dir, check=True)
subprocess.run(["uv", "sync", "--python", "3.12"], cwd=repo_dir, check=True)


In [ ]:
#@title 3. Run the Ko-BrowseComp evaluation
import os
import shlex
import subprocess

engine_key_env = {
    "tavily": "TAVILY_API_KEY",
    "perplexity": "PERPLEXITY_API_KEY",
    "perplexity-long": "PERPLEXITY_API_KEY",
    "brave": "BRAVE_API_KEY",
    "exa": "EXA_API_KEY",
}
provided_keys = {
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "TAVILY_API_KEY": TAVILY_API_KEY,
    "PERPLEXITY_API_KEY": PERPLEXITY_API_KEY,
    "BRAVE_API_KEY": BRAVE_API_KEY,
    "EXA_API_KEY": EXA_API_KEY,
}
for env_name, value in provided_keys.items():
    value = value.strip()
    if value:
        os.environ[env_name] = value

required = ["OPENAI_API_KEY", engine_key_env[SEARCH_ENGINE]]
missing = [env_name for env_name in required if not os.environ.get(env_name)]
if missing:
    raise ValueError("Missing required key(s): " + ", ".join(missing))

run_suffix = RUN_SUFFIX.strip() or "colab"
cmd = [
    "uv",
    "run",
    "--python",
    "3.12",
    "python",
    "search_evals/run_eval.py",
    f"search_engine={SEARCH_ENGINE}",
    f"model={MODEL}",
    "suite=ko_browsecomp",
    "hf_dataset_repo=prometheus-eval/k-browsecomp",
    f"hf_dataset_config={HF_DATASET_CONFIG}",
    "hf_dataset_split=test",
    f"grader_model={GRADER_MODEL}",
    f"dry_run={str(DRY_RUN).lower()}",
    f"max_workers={MAX_WORKERS}",
    f"run_suffix={run_suffix}",
    "rerun=true",
]
print("$", " ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, cwd=repo_dir, check=True)


In [ ]:
#@title 4. Show the score and per-question grading summary
import json
from pathlib import Path

import pandas as pd

run_name = f"{SEARCH_ENGINE}-{MODEL}_ko_browsecomp"
if run_suffix:
    run_name = f"{run_name}-{run_suffix}"

result_path = repo_dir / "runs" / "results" / f"{run_name}.json"
result = json.loads(result_path.read_text())
print(json.dumps(result, indent=2, ensure_ascii=False))
print(f"Result file: {result_path}")

rows = []
for task_path in sorted((repo_dir / "runs" / run_name).glob("*.json")):
    task = json.loads(task_path.read_text())
    datum = task.get("datum", {})
    grader = task.get("grader_result", {})
    rows.append(
        {
            "id": datum.get("id", task_path.stem),
            "score": 1 if grader.get("grade_type") == 1 else 0,
            "answer": datum.get("answer", ""),
            "grade_text": grader.get("grade_text", "")[:400],
        }
    )

pd.DataFrame(rows)
